# STKI RAG Demo
Demo Retrieval Augmented Generation (RAG) untuk mata kuliah STKI.
Project ini membandingkan 3 pendekatan retrieval semantik:
1. Prompt-based (in-context LLM reasoning)
2. Vector search (embedding + cosine similarity)
3. KL-Divergence (softmax probability distribution)
Menggunakan model Google Gemini untuk embedding dan generasi jawaban.

## Cell 1: Setup
Import libraries and initialize Gemini client

In [1]:
import os
import time
import hashlib
from typing import Iterable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dotenv import load_dotenv
from google import genai


# Load .env and get API key
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY tidak ditemukan. "
        "Isi file .env dengan GEMINI_API_KEY=key-mu."
    )

client = genai.Client(api_key=GEMINI_API_KEY)

## Cell 2: Helper Functions
Dua fungsi utama yang dipakai berulang kali di notebook ini:
- `get_embedding` — ubah teks menjadi vektor (dengan retry + fallback lokal)
- `cosine_sim` — hitung cosine similarity antar dua vektor

In [2]:
def get_embedding(text: str, retries: int = 4, base_delay: float = 1.0) -> list[float]:
    """Generate embedding menggunakan Gemini dengan retry."""

    for attempt in range(retries):
        try:
            resp = client.models.embed_content(
                model="gemini-embedding-001",
                contents=text,
            )
            return resp.embeddings[0].values

        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(base_delay * (2 ** attempt))
            else:
                raise RuntimeError(
                    f"Gagal membuat embedding setelah {retries} percobaan."
                ) from exc


def cosine_sim(a, b) -> float:
    """Cosine similarity between two vectors.
    
    Returns 0.0 if either vector is zero-length to avoid division by zero.
    """
    a, b = np.array(a), np.array(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def chunk_text(document: str) -> list[str]:
    """Chunking berdasarkan heading level 2"""

    sections = document.split("## ")

    chunks = []

    for section in sections[1:]:      # Mulai dari index 1
        chunks.append("## " + section.strip())

    return chunks

## Cell 3: Load Query and Documents
Tentukan query (pertanyaan) dan muat dokumen dari folder `dokumen/`.

In [3]:
query = "What time do I leave for school?"

def load_documents() -> dict[str, str]:
    """Read all markdown files from dokumen/ directory."""
    docs = {}
    for fname in ["D1.md"]:
        with open(os.path.join("dokumen", fname), encoding="utf-8") as f:
            docs[fname[:-3]] = f.read().strip()
    return docs

DOCS = load_documents()
chunked_docs = {
    
}

for doc_name, text in DOCS.items():
    chunks = chunk_text(text)
    for i, chunk in enumerate(chunks):
        chunk_name = f"{doc_name}_chunk_{i+1}"
        chunked_docs[chunk_name] = chunk
        
        
print("Query:", query)
print("Documents loaded:", list(chunked_docs.keys()))
for name, text in chunked_docs.items():
    print(f"--- {name} ---")
    print(text)
    print()

Query: What time do I leave for school?
Documents loaded: ['D1_chunk_1', 'D1_chunk_2', 'D1_chunk_3', 'D1_chunk_4']
--- D1_chunk_1 ---
## Morning Routine

I usually wake up at 5:30 AM every weekday. After waking up, I brush my teeth, take a shower, and prepare breakfast. I often eat bread, eggs, and drink a cup of coffee before starting my day. At around 6:30 AM, I leave home and travel to school by motorcycle.

--- D1_chunk_2 ---
## School Activities

My classes begin at 7:30 AM and usually finish at 3:00 PM. During school hours, I attend lectures, participate in laboratory sessions, complete assignments, and discuss projects with my classmates. I also spend some time in the library to review course materials before going home.

--- D1_chunk_3 ---
## Evening Routine

After arriving home, I take a short break and have dinner with my family. In the evening, I usually study programming, practice machine learning, or work on university projects. Before going to bed, I read a book or watch 

In [4]:
print(DOCS)

{'D1': '# Daily Routine\n\n## Morning Routine\n\nI usually wake up at 5:30 AM every weekday. After waking up, I brush my teeth, take a shower, and prepare breakfast. I often eat bread, eggs, and drink a cup of coffee before starting my day. At around 6:30 AM, I leave home and travel to school by motorcycle.\n\n## School Activities\n\nMy classes begin at 7:30 AM and usually finish at 3:00 PM. During school hours, I attend lectures, participate in laboratory sessions, complete assignments, and discuss projects with my classmates. I also spend some time in the library to review course materials before going home.\n\n## Evening Routine\n\nAfter arriving home, I take a short break and have dinner with my family. In the evening, I usually study programming, practice machine learning, or work on university projects. Before going to bed, I read a book or watch educational videos. I normally sleep at around 10:30 PM.\n\n## Weekend Activities\n\nOn weekends, my schedule is more flexible. I enjoy

## Cell 4: Vector Search
Langkah-langkah:
1. Embed semua dokumen menjadi vektor.
2. Embed query menjadi vektor.
3. Hitung cosine similarity antara query dan setiap dokumen.
4. Pilih dokumen dengan skor tertinggi.

In [5]:
doc_vecs = {}
for k, v in chunked_docs.items():
    try:
        doc_vecs[k] = get_embedding(v)
        print(f"Embedded {k}: {len(doc_vecs[k])} dimensions")
        table = pd.DataFrame({
            "Document" : k,
            "Embedding Value" : [doc_vecs[k]]
        })
        display(table)
        
    except Exception as exc:
        print(f"Skip dokumen {k} karena error: {exc}")

if not doc_vecs:
    raise RuntimeError("Tidak ada embedding dokumen yang berhasil dibuat.")

q_vec = get_embedding(query)
print(f"Query embedded: {len(q_vec)} dimensions")
print()

scores = {k: cosine_sim(q_vec, v) for k, v in doc_vecs.items()}
best = max(scores, key=scores.get)

print("Similarity scores:")
for k, v in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {k}: {v:.4f}")
print()
print(f"Best match: {best}")


Embedded D1_chunk_1: 3072 dimensions


,Document,Embedding Value
0,D1_chunk_1,"[-0.0029965248, 0.022050943, 0.019949187, -0.0..."


Embedded D1_chunk_2: 3072 dimensions


,Document,Embedding Value
0,D1_chunk_2,"[0.013931354, 0.017161366, 0.007971576, -0.036..."


Embedded D1_chunk_3: 3072 dimensions


,Document,Embedding Value
0,D1_chunk_3,"[0.032076564, 0.012565903, 0.014155079, -0.072..."


Embedded D1_chunk_4: 3072 dimensions


,Document,Embedding Value
0,D1_chunk_4,"[0.027821863, 0.020232225, -0.0081108585, -0.0..."


Query embedded: 3072 dimensions

Similarity scores:
  D1_chunk_1: 0.6727
  D1_chunk_2: 0.6616
  D1_chunk_3: 0.5920
  D1_chunk_4: 0.5350

Best match: D1_chunk_1


## Cell 5: RAG Answer Generation
Dengan dokumen terbaik dari cosine similarity, kirim ke Gemini untuk generate
jawaban terstruktur: ANSWER / SNIPPET / REASON.

In [6]:
METRIC = "cosine similarity"  # bisa diganti ke "KL-Divergence"

doc_text = chunked_docs[best]
scores_text = "".join([f"- {k}: {v:.4f}" for k, v in scores.items()])

prompt = (
    "You are a semantic Question Answering system."
    f'A user asked: "{query}"'
    f"Based on {METRIC}, the best matching document is {best}."
    f"Scores:{scores_text}"
    f"Content of {best}:{doc_text}"
    "Answer the query using ONLY the content above."
    "Format your answer as:"
    f"ANSWER: [direct answer based on {best}]"
    "SNIPPET: [1 relevant sentence from the document]"
    "REASON: [why this document is the best match in 1 sentence]"
)

print("Generating answer from", best, "...")
start_answer = time.time()
resp = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
)
elapsed_answer = time.time() - start_answer

# Parse structured output
lines = resp.text.strip().splitlines()
rag_result = {"answer": "", "snippet": "", "reason": ""}
current = ""
for line in lines:
    if line.startswith("ANSWER:"):
        current = "answer"
        rag_result["answer"] = line.replace("ANSWER:", "").strip()
    elif line.startswith("SNIPPET:"):
        current = "snippet"
        rag_result["snippet"] = line.replace("SNIPPET:", "").strip()
    elif line.startswith("REASON:"):
        current = "reason"
        rag_result["reason"] = line.replace("REASON:", "").strip()
    elif current == "answer" and line.strip():
        rag_result["answer"] += " " + line.strip()
    elif current == "reason" and line.strip():
        rag_result["reason"] += " " + line.strip()

print("ANSWER:", rag_result["answer"])
print("SNIPPET:", rag_result["snippet"])
print("REASON:", rag_result["reason"])
print(f"[Generated in {elapsed_answer:.2f}s]")

Generating answer from D1_chunk_1 ...
ANSWER: You leave for school at around 6:30 AM.
SNIPPET: At around 6:30 AM, I leave home and travel to school by motorcycle.
REASON: This document is the best match because it is the only chunk that contains specific information regarding the time the individual departs for school.
[Generated in 2.06s]
